# Host-response modelling — and controlling the depth confound

**Utility:** ViralScan's `hostresponse` module asks whether infected cells have a
distinct host expression program, using logistic regression with viral genes excluded
from the features. Its more important utility is **exposing and controlling the
sequencing-depth confound** that inflates naive host-response AUROCs.

This notebook reads the *committed* result files from the manuscript's matched-EBV
host-response run (`results/hostresponse_ebv_matched/`), so it runs without the heavy
input `.h5ad`.

## The headline metric

Across six seeds, an L2 logistic model predicted EBV-high status from host expression
at AUROC ≈ 0.87. Taken alone this looks like a strong host-response signature.

In [ ]:
from pathlib import Path
import pandas as pd

run = Path('..') / '..' / 'results' / 'hostresponse_ebv_matched'
if not run.exists():
    run = Path('results') / 'hostresponse_ebv_matched'  # if run from repo root

metrics = pd.read_csv(run / 'hostresponse_metrics.csv')
metrics[['virus', 'n_positive', 'auc_mean', 'auc_sd',
         'balanced_acc_mean', 'sensitivity_mean', 'specificity_mean']]

## The confound: the label tracks library size

The `>=10 corrected UMI` label is strongly correlated with host sequencing depth: a
deeper cell has more of *everything*, including viral UMIs. `depth_confounder.txt`
shows EBV-high prevalence rising monotonically across host-depth quintiles.

In [ ]:
print((run / 'depth_confounder.txt').read_text())

Read the key line: **depth alone** predicts the label at AUROC 0.967 within the
headline evaluation design — *higher than the 0.866 host-gene model itself*. So the
raw AUROC is largely a depth-tracking artifact, not depth-independent biology.

## Depth-controlled estimates bound the real signal

ViralScan reports depth-controlled analyses: a CPM-normalised, prevalence-matched
label, and a depth-matched case/control cohort. These bound the depth-independent
host-response signal at roughly **0.64–0.72** — much weaker than 0.87.

In [ ]:
for f in ['cpm_label_crosscheck.txt', 'depth_matched_reanalysis.txt']:
    p = run / f
    if p.exists():
        print('=' * 70)
        print(f)
        print('=' * 70)
        print(p.read_text())

## Depth-robust genes

Randomised Lasso stability selection retains features robust to moderate depth
confounding (depth-adjusted E-value ≥ 2).

In [ ]:
robust = run / 'depth_robust_genes.md'
print(robust.read_text() if robust.exists() else 'not present')

## In a real run

`hostresponse` runs on a completed sample directory plus the host-only AnnData
(`--host-h5ad`). Use `--label cpm` and `--depth-match` to blunt the depth confound:

```bash
viralscan hostresponse -o out/sample/ --host-h5ad host.h5ad \
    --label cpm \        # CPM-normalise the features (raw|cpm|fraction)
    --depth-match \      # restrict to a depth-matched case/control cohort
    --n-stab-iter 100    # randomised-Lasso stability selection
```

## Summary

- `hostresponse` associates host expression with viral burden — **but always read the
  AUROC against the reported depth-alone baseline.**
- A raw viral-UMI threshold tracks library size; use `--label cpm` and `--depth-match`
  for a depth-independent estimate.
- Here the honest signal is ≈ 0.64–0.72, not the headline 0.87.